# Approach 3: Explicit metal mask as a third input channel

Approach 1's dominant false-positive failure mode was the detector confusing other
metal hardware (screws/plates elsewhere in the slice) for real osteotomy sites.
Right now the model only receives raw brightness and has to infer "this bright blob
is metal, not bone" entirely on its own from a single grayscale channel duplicated
into R, G and B.

**The idea**: give the model an explicit metal mask as its own channel. Part 1 found
that metal saturates to the maximum representable intensity (2^15-1 in the raw HU
volume); the same holds in this dataset's exported 8-bit JPGs, where metal
consistently pushes pixels to their brightest values, clearly separated from bone.
A simple brightness threshold (>=220 out of 255, checked visually below) isolates
just the screws/plate, not bone. Replacing the usual R=G=B grayscale duplication
with **R=raw, G=metal mask, B=raw** hands the model this segmentation for free,
freeing it to spend its capacity learning the surrounding bone texture (the
probably-real distinguishing signal) rather than re-deriving "bright = metal" from
scratch in every layer.

Everything else is kept identical to Approach 1 (resize, not crop) for a clean,
single-variable comparison: same patient split, same model, same epochs/batch/seed.
Only the image channels change.

## Build the 3-channel dataset

`R = original grayscale`, `G = binary metal mask (pixel >= 220)`, `B = original
grayscale` (same duplication YOLO would otherwise do for G, just replaced with the
mask). Labels are untouched - this only changes pixel content, not box positions.

In [1]:
from pathlib import Path
import numpy as np
from PIL import Image

DATASET_DIR = Path("../dataset")
IMAGES_DIR = DATASET_DIR / "images"
OUT_DIR = DATASET_DIR / "images_metalmask"
OUT_DIR.mkdir(exist_ok=True)

METAL_THRESH = 220

n = 0
mask_fracs = []
for img_path in sorted(IMAGES_DIR.glob("*.jpg")):
    gray = np.array(Image.open(img_path).convert("L"))
    mask = (gray >= METAL_THRESH).astype(np.uint8) * 255
    rgb = np.stack([gray, mask, gray], axis=-1).astype(np.uint8)
    Image.fromarray(rgb).save(OUT_DIR / img_path.name, quality=95)
    mask_fracs.append(mask.mean() / 255)
    n += 1

print(f"Processed {n} images")
print(f"Mean fraction of pixels flagged as metal: {np.mean(mask_fracs):.4f}")
print(f"Images with zero metal pixels flagged: {sum(1 for f in mask_fracs if f == 0)}")

Processed 820 images
Mean fraction of pixels flagged as metal: 0.0027
Images with zero metal pixels flagged: 0


## Arrange into YOLO's expected layout

Same patient -> split mapping as Approach 1/2 (`dataset/splits.json`), and the
original (uncropped) labels, since box positions are unaffected by the channel
change.

In [2]:
import json
import shutil

split_map = json.load(open(DATASET_DIR / "splits.json"))
LABELS_DIR = DATASET_DIR / "labels"
YOLO_DIR = DATASET_DIR / "yolo_metalmask"

for split in ["train", "val", "test"]:
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

counts = {"train": 0, "val": 0, "test": 0}
for img_path in sorted(OUT_DIR.glob("*.jpg")):
    patient_id = img_path.stem.split("_")[0]
    split = split_map[patient_id]
    label_path = LABELS_DIR / f"{img_path.stem}.txt"
    shutil.copy2(img_path, YOLO_DIR / "images" / split / img_path.name)
    shutil.copy2(label_path, YOLO_DIR / "labels" / split / label_path.name)
    counts[split] += 1

print(f"Copied files into {YOLO_DIR.resolve()}")
print(counts)

Copied files into E:\Bone Union Detection\dataset\yolo_metalmask
{'train': 574, 'val': 123, 'test': 123}


In [3]:
data_yaml = f"""path: {YOLO_DIR.resolve().as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: osteotomy_site
"""

data_yaml_path = YOLO_DIR / "data.yaml"
data_yaml_path.write_text(data_yaml)
print(data_yaml_path.read_text())

path: E:/Bone Union Detection/dataset/yolo_metalmask
train: images/train
val: images/val
test: images/test

names:
  0: osteotomy_site



## Train

Identical configuration to Approach 1: 100 epoch budget, patience=20, imgsz=320
(resize, same as Approach 1 - not the Approach 2 crop), batch=32, `cache='ram'`,
seed=42. The only difference from Approach 1 is the metal-mask G channel.

In [4]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(data_yaml_path),
    epochs=100,
    patience=20,
    imgsz=320,
    batch=32,
    cache="ram",
    project="../runs",
    name="osteotomy_yolov8n_metalmask",
    seed=42,
    plots=False,  # avoid saving mosaics/prediction images that embed dataset content
)

New https://pypi.org/project/ultralytics/8.4.116 available  Update with 'pip install -U ultralytics'


Ultralytics 8.4.115  Python-3.11.9 torch-2.13.0+cpu CPU (11th Gen Intel Core i7-1185G7 @ 3.00GHz)


engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\dataset\yolo_metalmask\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=osteotomy_yolov8n_metalmask, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

Overriding model.yaml nc=80 with nc=1



                   from  n    params  module                                       arguments                     


...

Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 50% ━━━━━━────── 1/2 6.4s/it 1.9s<6.4s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.8s/it 3.7s

                   all        123        180      0.722      0.594      0.634       0.21


Speed: 0.4ms preprocess, 26.7ms inference, 0.0ms loss, 0.6ms postprocess per image


## Task 4 (metal-mask): Evaluate on the held-out test split

Same 123 test images, same 25 patients as Approach 1 and Approach 2.

In [5]:
best_weights = Path(results.save_dir) / "weights" / "best.pt"
print(f"Loading best checkpoint from {best_weights}")

test_model = YOLO(str(best_weights))
test_metrics = test_model.val(data=str(data_yaml_path), split="test", imgsz=320, plots=False)

print("\nTest set (123 images, 25 patients) - metal-mask model:")
print(f"Precision: {test_metrics.box.mp:.3f}")
print(f"Recall:    {test_metrics.box.mr:.3f}")
print(f"mAP50:     {test_metrics.box.map50:.3f}")
print(f"mAP50-95:  {test_metrics.box.map:.3f}")

Loading best checkpoint from E:\Bone Union Detection\notebooks\runs\runs\osteotomy_yolov8n_metalmask\weights\best.pt
Ultralytics 8.4.115  Python-3.11.9 torch-2.13.0+cpu CPU (11th Gen Intel Core i7-1185G7 @ 3.00GHz)


Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 3.00.6 MB/s, size: 38.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/


val: Scanning E:\Bone Union Detection\dataset\yolo_metalmask\labels\test... 20 images, 0 backgrounds, 0 corrupt: 16% ━╸────────── 20/123 59.0it/s 0.1s<1.7s

val: Scanning E:\Bone Union Detection\dataset\yolo_metalmask\labels\test... 44 images, 0 backgrounds, 0 corrupt: 35% ━━━━──────── 44/123 99.3it/s 0.2s<0.8s

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 87% ━━━━━━━━━━── 7/8 1.9it/s 3.5s<0.5s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s

                   all        123        173      0.656      0.538      0.502       0.14


Speed: 0.6ms preprocess, 23.9ms inference, 0.0ms loss, 0.9ms postprocess per image



Test set (123 images, 25 patients) - metal-mask model:
Precision: 0.656
Recall:    0.538
mAP50:     0.502
mAP50-95:  0.140


## Task 5 (metal-mask): Qualitative results and per-image analysis

Same IoU>=0.5 matching methodology as Approaches 1 and 2. Overlays saved to
`qualitative_results_metalmask/` (gitignored, not committed).

In [6]:
from PIL import Image, ImageDraw

OUT_QUAL_DIR = Path("../qualitative_results_metalmask")
OUT_QUAL_DIR.mkdir(exist_ok=True)

test_images = sorted((YOLO_DIR / "images" / "test").glob("*.jpg"))


def load_gt_boxes(label_path, img_w, img_h):
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        _, xc, yc, w, h = [float(p) for p in line.split()]
        boxes.append([
            (xc - w / 2) * img_w, (yc - h / 2) * img_h,
            (xc + w / 2) * img_w, (yc + h / 2) * img_h,
        ])
    return boxes


def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


IOU_THRESH = 0.5
per_image_results = []

for img_path in test_images:
    label_path = YOLO_DIR / "labels" / "test" / f"{img_path.stem}.txt"
    w, h = Image.open(img_path).size
    gt_boxes = load_gt_boxes(label_path, w, h)

    pred = test_model.predict(str(img_path), imgsz=320, conf=0.25, verbose=False)[0]
    pred_boxes = pred.boxes.xyxy.cpu().numpy().tolist() if len(pred.boxes) else []
    pred_confs = pred.boxes.conf.cpu().numpy().tolist() if len(pred.boxes) else []

    matched_gt, matched_pred = set(), set()
    for pi, pb in enumerate(pred_boxes):
        best_iou, best_gi = 0, -1
        for gi, gb in enumerate(gt_boxes):
            if gi in matched_gt:
                continue
            v = iou(pb, gb)
            if v > best_iou:
                best_iou, best_gi = v, gi
        if best_iou >= IOU_THRESH:
            matched_gt.add(best_gi)
            matched_pred.add(pi)

    n_gt, n_pred, n_tp = len(gt_boxes), len(pred_boxes), len(matched_gt)
    per_image_results.append({
        "path": img_path, "gt_boxes": gt_boxes,
        "pred_boxes": pred_boxes, "pred_confs": pred_confs,
        "n_gt": n_gt, "n_pred": n_pred, "n_tp": n_tp,
        "n_fn": n_gt - n_tp, "n_fp": n_pred - len(matched_pred),
        "recall": n_tp / n_gt if n_gt > 0 else None,
    })

by_group = {}
for r in per_image_results:
    key = "1 box" if r["n_gt"] == 1 else ("2+ boxes" if r["n_gt"] >= 2 else "0 boxes")
    by_group.setdefault(key, []).append(r)

print("=== Per-image recall by ground-truth box count (test split, metal-mask model) ===")
for key, items in sorted(by_group.items()):
    n_images = len(items)
    total_gt = sum(r["n_gt"] for r in items)
    total_tp = sum(r["n_tp"] for r in items)
    perfect = sum(1 for r in items if r["recall"] == 1.0)
    print(f"{key}: {n_images} images, {total_gt} GT boxes, "
          f"box-level recall={total_tp/total_gt:.3f}, "
          f"{perfect}/{n_images} images fully detected ({perfect/n_images:.1%})")

total_fp = sum(r["n_fp"] for r in per_image_results)
total_pred = sum(r["n_pred"] for r in per_image_results)
print(f"\nTotal predicted boxes: {total_pred}, false positives: {total_fp} "
      f"({total_fp/total_pred:.1%} of all predictions)")

perfect_cases = [r for r in per_image_results if r["n_gt"] > 0 and r["recall"] == 1.0 and r["n_fp"] == 0]
miss_cases = sorted([r for r in per_image_results if r["n_fn"] > 0], key=lambda r: -r["n_fn"])
fp_cases = sorted([r for r in per_image_results if r["n_fp"] > 0], key=lambda r: -r["n_fp"])
print(f"\nPerfect-detection images: {len(perfect_cases)}/{len(per_image_results)}")
print(f"Images with >=1 missed box: {len(miss_cases)}/{len(per_image_results)}")
print(f"Images with >=1 false positive: {len(fp_cases)}/{len(per_image_results)}")


def draw_overlay(r, out_path):
    img = Image.open(r["path"]).convert("RGB")
    draw = ImageDraw.Draw(img)
    for gb in r["gt_boxes"]:
        draw.rectangle(gb, outline=(0, 255, 0), width=2)
    for pb, conf in zip(r["pred_boxes"], r["pred_confs"]):
        draw.rectangle(pb, outline=(255, 0, 0), width=2)
        draw.text((pb[0], max(0, pb[1] - 10)), f"{conf:.2f}", fill=(255, 0, 0))
    img.save(out_path, quality=95)


for category, cases in [("success", perfect_cases[:3]), ("missed_detection", miss_cases[:3]), ("false_positive", fp_cases[:3])]:
    for i, r in enumerate(cases):
        out_path = OUT_QUAL_DIR / f"{category}_{i}_{r['path'].stem}.jpg"
        draw_overlay(r, out_path)
        print(f"Saved {category} example: {out_path.name} "
              f"(gt={r['n_gt']}, tp={r['n_tp']}, fn={r['n_fn']}, fp={r['n_fp']})")

=== Per-image recall by ground-truth box count (test split, metal-mask model) ===
1 box: 87 images, 87 GT boxes, box-level recall=0.529, 46/87 images fully detected (52.9%)
2+ boxes: 36 images, 86 GT boxes, box-level recall=0.535, 8/36 images fully detected (22.2%)

Total predicted boxes: 131, false positives: 39 (29.8% of all predictions)

Perfect-detection images: 50/123
Images with >=1 missed box: 69/123
Images with >=1 false positive: 31/123
Saved success example: success_0_1004_1_75.jpg (gt=1, tp=1, fn=0, fp=0)
Saved success example: success_1_1004_1_79.jpg (gt=1, tp=1, fn=0, fp=0)
Saved success example: success_2_1004_2_134.jpg (gt=1, tp=1, fn=0, fp=0)
Saved missed_detection example: missed_detection_0_920_1_31.jpg (gt=3, tp=0, fn=3, fp=2)
Saved missed_detection example: missed_detection_1_920_1_53.jpg (gt=3, tp=0, fn=3, fp=0)
Saved missed_detection example: missed_detection_2_120_2_117.jpg (gt=2, tp=0, fn=2, fp=0)
Saved false_positive example: false_positive_0_163_1_46.jpg (gt=2